# embeddings of our dataset

Creates one 768-dimensional embedding per publication from its title and abstract
using the local Ollama `nomic-embed-text` model. The notebook prepares the text,
tests the resumable workflow on a small sample, and validates the final artifact.

This notebooks assumed: 
- You have the embedding model running. Please follow instructions here: https://ollama.com/library/nomic-embed-text
- You have access to both title and abstracts of the dataset. Complying with the Web of Science terms of use, abstracts are not shared in our dataset. We recommend to download those manually and set up before running this notebook at scale 

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import shutil

import numpy as np
import pandas as pd

from data_helpers import results_config
from data_helpers.prep import embeddings_prep


## 1. Configuration


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
EMBEDDINGS_CONFIG = RESULTS_CONFIG["embeddings_prep"]

OUTPUT_DIR = PROJECT_ROOT / EMBEDDINGS_CONFIG["output_directory"]
DATASET_PATH = OUTPUT_DIR / "dataset.parquet"
ABSTRACTS_PATH = OUTPUT_DIR / "dataset_abstracts.parquet"
EMBEDDINGS_PATH = OUTPUT_DIR / "embeddings.parquet"
MANIFEST_PATH = OUTPUT_DIR / "embeddings_manifest.json"
CACHE_DIR = OUTPUT_DIR / EMBEDDINGS_CONFIG["cache_subdirectory"]
PILOT_CACHE_DIR = OUTPUT_DIR / "_embeddings_cache_pilot"

EMBEDDINGS_CONFIG


{'description': 'One embedding vector per publication (title + abstract) via a local Ollama nomic-embed-text endpoint.',
 'endpoint': 'http://localhost:11434/api/embed',
 'model': 'nomic-embed-text',
 'text_prefix': 'search_document: ',
 'batch_size': 64,
 'shard_size': 5000,
 'request_timeout_seconds': 120,
 'max_retries': 5,
 'cache_subdirectory': '_embeddings_cache',
 'output_directory': 'notebooks/data_processing/outputs/04_dataset'}

## 2. Build the id/UT → text mapping

Joins titles and abstracts on `UT`, applies the configured text prefix, and checks
that every publication is retained.


In [3]:
dataset = pd.read_parquet(DATASET_PATH, columns=["id", "UT", "title"])
abstracts = pd.read_parquet(ABSTRACTS_PATH, columns=["UT", "abstract"])
texts = embeddings_prep.build_embedding_texts(
    dataset, abstracts, prefix=EMBEDDINGS_CONFIG["text_prefix"]
)
assert len(texts) == len(dataset)
assert texts["id"].is_unique and texts["UT"].is_unique

char_lengths = texts["text"].str.len()
word_counts = texts["text"].str.split().str.len()
length_summary = pd.DataFrame(
    {
        "chars": char_lengths.describe(percentiles=[0.5, 0.9, 0.99]),
        "words": word_counts.describe(percentiles=[0.5, 0.9, 0.99]),
    }
)
print(f"{len(texts):,} publications mapped to embedding text.")
display(length_summary)
display(texts.head(2))


605,199 publications mapped to embedding text.


,chars,words
count,605199.000000,605199.000000
mean,1804.378018,260.775282
std,473.730328,69.670720
min,112.000000,16.000000
50%,1774.000000,257.000000
90%,2354.000000,341.000000
99%,3124.000000,460.000000
max,19959.000000,2564.000000


,id,UT,text
0,1410940900001,WOS:001410940900001,search_document: Skeletal deformities in Sparu...
1,1498294300001,WOS:001498294300001,search_document: Size-dependent effects of oxo...


## 3. Pilot run (small N, against the live Ollama endpoint)

Embeds 300 publications in two passes to test checkpointing and resume behavior.


In [4]:
PILOT_N = 300
PILOT_FIRST_PASS = 150
pilot_texts = texts.head(PILOT_N)

if PILOT_CACHE_DIR.exists():
    shutil.rmtree(PILOT_CACHE_DIR)

first_pass_stats = embeddings_prep.run_embedding_build(
    texts=pilot_texts,
    cache_dir=PILOT_CACHE_DIR,
    endpoint=EMBEDDINGS_CONFIG["endpoint"],
    model=EMBEDDINGS_CONFIG["model"],
    batch_size=EMBEDDINGS_CONFIG["batch_size"],
    shard_size=64,
    timeout=EMBEDDINGS_CONFIG["request_timeout_seconds"],
    max_retries=EMBEDDINGS_CONFIG["max_retries"],
    limit=PILOT_FIRST_PASS,
)
first_pass_stats



  0%|          | 0/150 [00:00<?, ?doc/s]


 43%|████▎     | 64/150 [00:02<00:03, 21.57doc/s]


 85%|████████▌ | 128/150 [00:05<00:00, 22.54doc/s]


100%|██████████| 150/150 [00:06<00:00, 22.05doc/s]


100%|██████████| 150/150 [00:06<00:00, 22.07doc/s]

{'remaining_before_run': 150,
 'processed': 150,
 'errored': 0,
 'elapsed_seconds': 6.798347375006415}

In [5]:
second_pass_stats = embeddings_prep.run_embedding_build(
    texts=pilot_texts,
    cache_dir=PILOT_CACHE_DIR,
    endpoint=EMBEDDINGS_CONFIG["endpoint"],
    model=EMBEDDINGS_CONFIG["model"],
    batch_size=EMBEDDINGS_CONFIG["batch_size"],
    shard_size=64,
    timeout=EMBEDDINGS_CONFIG["request_timeout_seconds"],
    max_retries=EMBEDDINGS_CONFIG["max_retries"],
)
assert second_pass_stats["remaining_before_run"] == PILOT_N - PILOT_FIRST_PASS, (
    "Resume did not correctly skip the already-embedded first pass."
)
done_ids, _ = embeddings_prep.scan_cache(PILOT_CACHE_DIR)
assert len(done_ids) == PILOT_N

ms_per_doc = first_pass_stats["elapsed_seconds"] / max(first_pass_stats["processed"], 1) * 1000
print(
    f"First pass: {first_pass_stats['processed']} docs in "
    f"{first_pass_stats['elapsed_seconds']:.1f}s ({ms_per_doc:.1f} ms/doc)."
)
print(
    f"Resumed pass correctly skipped the first {PILOT_FIRST_PASS} ids and embedded "
    f"only the remaining {second_pass_stats['processed']} "
    f"({second_pass_stats['elapsed_seconds']:.1f}s)."
)
second_pass_stats



  0%|          | 0/150 [00:00<?, ?doc/s]


 43%|████▎     | 64/150 [00:02<00:03, 23.23doc/s]


 85%|████████▌ | 128/150 [00:05<00:00, 23.11doc/s]


100%|██████████| 150/150 [00:06<00:00, 23.05doc/s]


100%|██████████| 150/150 [00:06<00:00, 23.07doc/s]

First pass: 150 docs in 6.8s (45.3 ms/doc).
Resumed pass correctly skipped the first 150 ids and embedded only the remaining 150 (6.5s).


{'remaining_before_run': 150,
 'processed': 150,
 'errored': 0,
 'elapsed_seconds': 6.5025006249779835}

### Compact the pilot shards and sanity-check the embeddings

Combines the pilot shards, validates vector dimensions, and checks sample cosine
similarities.


In [6]:
pilot_manifest = embeddings_prep.compact_and_validate(
    texts=pilot_texts,
    cache_dir=PILOT_CACHE_DIR,
    output_path=PILOT_CACHE_DIR / "embeddings_pilot.parquet",
    manifest_path=PILOT_CACHE_DIR / "embeddings_pilot_manifest.json",
    model=EMBEDDINGS_CONFIG["model"],
    prefix=EMBEDDINGS_CONFIG["text_prefix"],
    endpoint=EMBEDDINGS_CONFIG["endpoint"],
    sources={"dataset": DATASET_PATH, "dataset_abstracts": ABSTRACTS_PATH},
    repository_root=PROJECT_ROOT,
)
pilot_embeddings = pd.read_parquet(PILOT_CACHE_DIR / "embeddings_pilot.parquet")
dims = pilot_embeddings["embedding"].map(len)
assert dims.nunique() == 1 and dims.iloc[0] == 768, "Unexpected embedding dimensionality."
print(f"Pilot compacted {len(pilot_embeddings):,} embeddings, dim={dims.iloc[0]}.")


def cosine(a, b) -> float:
    a, b = np.asarray(a, dtype="float64"), np.asarray(b, dtype="float64")
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


sample = (
    pilot_embeddings.merge(texts[["id", "text"]], on="id")
    .sample(6, random_state=0)
    .reset_index(drop=True)
)
labels = sample["text"].str.slice(0, 60)
sims = pd.DataFrame(
    [[cosine(sample["embedding"][i], sample["embedding"][j]) for j in range(len(sample))] for i in range(len(sample))],
    index=labels,
    columns=labels,
)
display(sims.round(3))

full_run_seconds = (len(texts) - PILOT_N) * (
    first_pass_stats["elapsed_seconds"] / max(first_pass_stats["processed"], 1)
)
print(
    f"Extrapolated full-run ETA for the remaining {len(texts) - PILOT_N:,} "
    f"publications: ~{full_run_seconds / 3600:.1f} hours, sequential single-worker."
)


Pilot compacted 300 embeddings, dim=768.


text,search_document: Exploring the complex interactions between,search_document: Effects of varying levels of dietary crude,search_document: Impacts of present heat waves in freshwater,search_document: Extra-simplified eco-viticulture increases,search_document: Model projections indicate substantial decr,search_document: Different pigmentation pattern formation be
text,,,,,,
search_document: Exploring the complex interactions between,1.000,0.658,0.626,0.610,0.572,0.659
search_document: Effects of varying levels of dietary crude,0.658,1.000,0.646,0.588,0.529,0.669
search_document: Impacts of present heat waves in freshwater,0.626,0.646,1.000,0.588,0.652,0.761
search_document: Extra-simplified eco-viticulture increases,0.610,0.588,0.588,1.000,0.684,0.596
search_document: Model projections indicate substantial decr,0.572,0.529,0.652,0.684,1.000,0.565
search_document: Different pigmentation pattern formation be,0.659,0.669,0.761,0.596,0.565,1.000


Extrapolated full-run ETA for the remaining 604,899 publications: ~7.6 hours, sequential single-worker.


## 4. Full run (outside this notebook)

Run the full build as a background process. Rerunning the command resumes from the
existing shard cache.

```bash
cd <Your root>
nohup venv/bin/python -m data_helpers.prep.embeddings_prep \
    > embeddings_run.log 2>&1 &
```

Check progress:

```bash
tail -f embeddings_run.log
ls notebooks/data_processing/outputs/04_dataset/_embeddings_cache | wc -l
```

On completion, the script writes `embeddings.parquet` and
`embeddings_manifest.json`.


## 5. Load, validate, and report the completed artifact


In [7]:
if EMBEDDINGS_PATH.exists():
    embeddings = pd.read_parquet(EMBEDDINGS_PATH)
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    assert len(embeddings) == len(texts)
    assert set(embeddings["id"]) == set(texts["id"])
    assert embeddings["embedding"].map(len).nunique() == 1
    print(f"Loaded {len(embeddings):,} embeddings, dim={manifest['embedding_dim']}.")
    display(pd.Series(manifest["rows"], name="publications"))
else:
    done_ids, _ = (
        embeddings_prep.scan_cache(CACHE_DIR) if CACHE_DIR.exists() else (set(), 0)
    )
    print(
        f"embeddings.parquet not built yet. {len(done_ids):,}/{len(texts):,} "
        f"publications embedded in the resumable cache so far ({CACHE_DIR})."
    )
    print(
        "Run the background command in section 4, then re-run this cell (or "
        "`python -m data_helpers.prep.embeddings_prep` again) to compact once complete."
    )


embeddings.parquet not built yet. 0/605,199 publications embedded in the resumable cache so far (/Users/hunain/d/coding-projects/biodiversity/notebooks/data_processing/outputs/04_dataset/_embeddings_cache).
Run the background command in section 4, then re-run this cell (or `python -m data_helpers.prep.embeddings_prep` again) to compact once complete.


## Handoff contract

`embeddings.parquet` contains `id`, `UT`, `embedding` (`list<float32>[768]`), and
`model`. The manifest records configuration, row counts, and any missing IDs.

Use the `search_query: ` prefix when embedding queries for comparison.
